## SP-1 Training — PaliGemma QLoRA on IAM-line
Runs on a Colab **T4** GPU. This notebook is thin glue over the tested `htr_sp1` package: load data, load the 4-bit model + LoRA, sanity-overfit gate, full fine-tune (resumable), evaluate CER/WER, export to the Hub, and a final reload-validation gate.

### 1. Install pinned deps & mount Drive
Drive holds checkpoints so a Colab disconnect doesn't lose hours of training.

In [ ]:
# Install the exact pinned versions and mount Drive so checkpoints survive disconnects.
!pip install -q -r /content/requirements.txt
from google.colab import drive
drive.mount('/content/drive')
import os
# Point training output at Drive so a disconnect doesn't lose hours of work.
os.environ["HTR_OUTPUT_DIR"] = "/content/drive/MyDrive/htr_sp1/outputs"
os.environ["HTR_HUB_REPO_ID"] = "YOUR_HF_USERNAME/paligemma-iam-line-qlora"  # <-- set this

### 2. Auth, import the package, seed everything

In [ ]:
# Make the package importable, authenticate to the Hub, and seed everything.
import sys; sys.path.insert(0, "/content/src")
from huggingface_hub import login; login()  # paste an HF token with write access
from htr_sp1 import config, data, model as model_mod, train, inference, evaluate, export
config.set_seed()

### 3. Load the IAM-line dataset
Official train/validation/test splits. Peek at one record to confirm image + text look right.

In [ ]:
# Load the official IAM-line splits; peek at one record to confirm image+text look right.
ds = data.load_iam_splits()
print(ds)
sample = ds["train"][0]; print(repr(sample["text"])); sample["image"]

### 4. Load PaliGemma in 4-bit + attach LoRA
Print trainable params to confirm ONLY the small adapter trains.

In [ ]:
# Load PaliGemma in 4-bit and attach the LoRA adapter. Print trainable params to confirm
# ONLY the small adapter trains (should be a tiny fraction of total params).
model, processor = model_mod.load_trainable_model()
model.print_trainable_parameters()

### 5. Sanity gate — overfit 2 samples
**If the training loss does NOT drop to ~0, STOP and fix `collate` before wasting T4 hours.** This is where the one risky integration point (batch collation) gets validated against real PaliGemma shapes.
> **If you hit a shape/padding error here:** the `collate` function in `htr_sp1/train.py` is the known risky integration point. `build_training_example` returns tensors with a leading batch dimension `[1, seq_len]`; `processor.tokenizer.pad` expects `[seq_len]` per example. If padding fails, squeeze the batch dim in `collate` — this is exactly the kind of fix the sanity gate exists to surface.

In [ ]:
# Train on a tiny 2-record slice for a few steps; loss should collapse toward 0.
tiny = ds["train"].select(range(2))
_ = train.run_training(model, processor, tiny, tiny,
                       output_dir="/content/sanity_check")
# Manually confirm the printed training loss trended to ~0 before continuing.

### 6. Full fine-tune
Resumes automatically from the latest Drive checkpoint if Colab disconnected.

In [ ]:
# NOTE: `model` was lightly warmed up by the sanity gate above (a few steps on 2 samples).
# The effect is negligible; the full run continues training the same adapter from here.
# Full fine-tune on the official splits; auto-resumes from Drive if Colab disconnected.
model = train.run_training(model, processor, ds["train"], ds["validation"],
                           output_dir=os.environ["HTR_OUTPUT_DIR"])

### 7. Evaluate on the test split (the M1 baseline numbers)

In [ ]:
# Bind the model into the inference contract and compute the baseline CER/WER (the M1 numbers).
transcribe = lambda img: inference.generate_transcription(model, processor, img)
report = evaluate.evaluate_split(ds["test"], transcribe)
print("TEST mean CER:", round(report["mean_cer"], 2), "  mean WER:", round(report["mean_wer"], 2))

### 8. Save metrics for the thesis appendix (Bab 4)

In [ ]:
# Persist per-sample + summary metrics to Drive for the Bab 4 appendix table.
import json, os
# Make sure the target folder exists even if this cell is run in isolation.
os.makedirs("/content/drive/MyDrive/htr_sp1", exist_ok=True)
with open("/content/drive/MyDrive/htr_sp1/test_metrics.json", "w") as f:
    json.dump(report, f, indent=2)

### 9. Export — push adapter + merged to the Hub (private)

In [ ]:
base = os.environ["HTR_HUB_REPO_ID"]
# Definition of done: write the adapter + merged model to disk FIRST, then push. If the push
# fails (e.g. token/network), the run is NOT lost — re-push later from disk with
# `python scripts/repush_sp1.py --output-dir $HTR_OUTPUT_DIR --hub-repo <base> --compute-dtype float16`.
# This notebook trains in fp16 on a Colab T4, so the merge uses float16.
status = export.save_and_push(
    model, processor,
    output_dir=os.environ["HTR_OUTPUT_DIR"], hub_repo=base, compute_dtype="float16",
)
print("Saved adapter ->", status["adapter_dir"], "| merged ->", status["merged_dir"])
if status["pushed"]:
    print("Pushed:", export.adapter_repo_id(base), "and", export.merged_repo_id(base))
else:
    print("PUSH FAILED:", status["error"],
          "\nArtifacts are safe on disk — re-push with scripts/repush_sp1.py")

### 10. Validation gate — reload fresh from the Hub
Reload the MERGED model with NO local state and transcribe a few test images. This is SP-1's definition of done.

In [ ]:
# Reload the MERGED model from the Hub (no local state) and confirm it transcribes sanely.
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor
rid = export.merged_repo_id(base)
v_model = PaliGemmaForConditionalGeneration.from_pretrained(rid, device_map="auto")
v_proc = PaliGemmaProcessor.from_pretrained(rid)
for i in range(3):
    img = ds["test"][i]["image"]
    print("GT :", ds["test"][i]["text"])
    print("OUT:", inference.generate_transcription(v_model, v_proc, img))